# 07 – Quick-Tuning (Widget-Interface)

Bedient `Src.backtest.tuning.tune_basket_params` über ein ipywidgets-Panel — EIN Abschnitt pro
Strategie (`pairwise_zscore`, `garch_wilks`). Pro Parameter ist EINZELN wählbar, ob er

- **fix** auf einen Wert gesetzt wird (Checkbox "tunen" AUS, Fix-Feld aktiv), oder
- **gesucht** wird, mit einem einstellbaren Bereich/einer Auswahl (Checkbox "tunen" AN,
  Bereichs-/Auswahl-Feld aktiv) statt des Defaults aus `DEFAULT_SEARCH_SPACE`.

Funktionales Pendant zu `notebooks/03_Tradeing/quick_tuning.ipynb`, hier auf der aktuellen Engine
(`src/backtest/tuning.py`, `search_space`-Parameter) statt der alten, strategie-eigenen Tuner
(`quick_tune_ewma.py`/`quick_tune_bollinger.py`/`quick_tune_garch.py`).

Anders als das alte Notebook läuft hier **eine gemeinsame Walk-Forward-Suche** über
`tune_basket_params` (Optuna/TPE, persistenter SQLite-Storage unter
`data/processed/tuning/optuna_walkforward.db`) statt drei komplett getrennter Tuner-Module — jede
Strategie unterscheidet sich nur durch `STRATEGY_NAME` und ihren eigenen `search_space`
(`pairwise_zscore`: `entry_z`/`exit_z`/`max_holding_half_life_multiplier`; `garch_wilks`:
`wilks_p_content`/`wilks_confidence`/`max_holding_half_life_multiplier`).

Jeder Tuning-Lauf speichert automatisch `best_params.json` + `trials_<basket>.parquet` unter
`data/processed/tuning/<run_id>/` (`Src.backtest.tuning.save_tuning_results`) und zeigt das Ergebnis
direkt im Panel an. Beim Klick auf "Tuning starten" wird zusätzlich automatisch ein
`optuna-dashboard` für die jeweilige Study gestartet (falls noch nicht laufend) und im Browser
geöffnet -- Pairwise auf Port 8080, GARCH-Wilks auf 8082 (siehe `Src.backtest.tuning.DASHBOARD_PORTS`
und `ensure_dashboard_running`, portiert aus dem alten `tuning_common.py`). Beide können
gleichzeitig laufen, ein bereits laufendes Dashboard wird nicht doppelt gestartet. Voraussetzung:
`pip install optuna-dashboard` -- fehlt das Kommando, wird nur gewarnt, das Tuning läuft trotzdem
normal weiter.

**Nicht direkt ausführen ohne Anpassung.** `RUN_LABEL_PREFIX` in Abschnitt 1 ändern, dann normal
ausführen -- die Widget-Panels selbst benötigen keine weitere Code-Änderung.

## Optuna-Dashboard manuell starten

Wird beim Klick auf "Tuning starten" bereits automatisch versucht (`ensure_dashboard_running`) --
falls das aus irgendeinem Grund nicht klappt (z.B. Prozess wurde beendet, `optuna-dashboard`
war zum Zeitpunkt des Klicks noch nicht installiert), im Terminal manuell nachholen:

```bash
pip install optuna-dashboard   # einmalig, falls noch nicht installiert

# Pairwise-Z-Score-Study (Port 8080):
optuna-dashboard sqlite:///data/processed/tuning/optuna_walkforward.db --host localhost --port 8080

# GARCH-Wilks-Study (Port 8082, gleiche DB-Datei -- alle Studies liegen in derselben SQLite-Datei,
# nur unterschiedliche study_name je Strategie/Basket):
optuna-dashboard sqlite:///data/processed/tuning/optuna_walkforward.db --host localhost --port 8082
```

Danach im Browser `http://localhost:8080/` bzw. `http://localhost:8082/` öffnen. Ein bereits
laufendes Dashboard mit `Strg+C` im jeweiligen Terminal beenden, bevor man es manuell neu startet
(sonst meldet `ensure_dashboard_running` beim nächsten Klick "läuft bereits" und lässt den alten
Prozess unangetastet).

## Was bedeutet `min_trades_per_fold`?

Reiner Qualitäts-**Filter**, kein Signalparameter: `mean(fold_returns) - std(fold_returns)`
(die eigentliche Zielgröße der Suche, siehe `Src.backtest.tuning._objective_metric`) würde allein
Kandidaten **pervers belohnen**, die kaum handeln — viele Folds mit ~0 Trades ziehen sowohl die
Rendite jedes Folds als auch die Streuung zwischen den Folds künstlich Richtung 0, was den
`mean - std`-Score anhebt, ohne dass die Strategie tatsächlich besser wäre (ein Parametersatz, der
fast nie einen Trade auslöst, "verliert" auf dem Papier kaum, weil er kaum etwas tut).

`min_trades_per_fold` verlangt deshalb, dass ein Kandidat über **alle** Folds zusammen mindestens
`min_trades_per_fold * n_folds` Trades erzeugt (Default 1.0 = im Schnitt mindestens 1 Trade pro
Fold) — Kandidaten darunter werden per `optuna.TrialPruned()` verworfen und zählen nicht zum
Ergebnis. Ist die Suche zu streng eingestellt (z.B. `entry_z`-Bereich zu hoch, `wilks_confidence`
zu nah an 1.0), kann es passieren, dass **kein** Trial die Schwelle erreicht — dann bricht
`tune_basket_params` mit `RuntimeError` ab; entweder `min_trades_per_fold` senken oder `n_trials`
erhöhen (mehr Versuche, einen handelnden Kandidaten zu finden).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display

from Src.data import available_data_range, load_ohlcv_panel, load_symbol_map
from Src.backtest.cliques import CliqueTimeline
from Src.backtest.config import BasketParams, split_calibration_backtest
from Src.backtest import tuning as bt_tuning
from Src.config import BINANCE_FEE_PRESETS_BPS


def _resolve_fee_bps(fee_preset: str) -> float:
    """Preset-Name (siehe Src.config.BINANCE_FEE_PRESETS_BPS) -> bps-Wert,
    fuer den Fee-Preset-Dropdown in jedem Strategie-Panel."""
    return BINANCE_FEE_PRESETS_BPS[fee_preset]


print("Imports OK")

Imports OK


## 1. Run-Konfiguration

In [2]:
RUN_LABEL_PREFIX = "quick_tuning"  # Praefix fuer gespeicherte Tuning-Laeufe (Abschnitt 5/6)

BACKTEST_FRACTION = 0.2  # gleiches Muster wie in _template.ipynb -- Tuning laeuft NUR auf CALIBRATION_RANGE

CLIQUE_VARIANT = "raw_24h"  # raw|residual x 6h|12h|24h, siehe data/processed/cliques/

FEE_PRESETS = ("standard", "bnb_discount", "vip1")  # siehe Src.config.BINANCE_FEE_PRESETS_BPS

## 2. Universum & Zeitraum

Wie im alten `quick_tuning.ipynb`: Zeitraum (`DATA_START`/`DATA_END`) und Baskets werden dynamisch
aus den tatsächlich vorhandenen Daten ermittelt (`Src.data.available_data_range` über die OHLCV-
Rohdaten, `timeline.coverage()` über den Clique-Cache) statt hart codiert -- ändert sich der
Datenbestand (neue Monate, neue Cliquen-Läufe), passt sich dieses Notebook automatisch an, ohne
Konstanten manuell nachzuziehen.

In [3]:
id_to_symbol, symbol_to_id = load_symbol_map()

DATA_START, DATA_END = available_data_range()
print(f"Verfuegbarer Zeitraum: {DATA_START.date()} -> {DATA_END.date()}")

panel = load_ohlcv_panel(start=DATA_START, end=DATA_END, fields=("open", "high", "low", "close"))
print(f"Panel: {panel['close'].shape[0]:,} Kerzen x {panel['close'].shape[1]} Symbole")

timeline = CliqueTimeline.from_cache(variant=CLIQUE_VARIANT, id_to_symbol=id_to_symbol, start=DATA_START, end=DATA_END)
BASKET_LABELS = sorted(timeline.coverage()["basket"].tolist())
print(f"Baskets in '{CLIQUE_VARIANT}': {BASKET_LABELS}")

CALIBRATION_RANGE, BACKTEST_RANGE = split_calibration_backtest(DATA_START, DATA_END, fraction=BACKTEST_FRACTION)
print(f"Calibration (Tuning): {CALIBRATION_RANGE.start.date()} -> {CALIBRATION_RANGE.end.date()}")

Verfuegbarer Zeitraum: 2024-07-23 -> 2026-07-22
Panel: 210,240 Kerzen x 47 Symbole
Baskets in 'raw_24h': ['Basket 1', 'Basket 2', 'Basket 3', 'Basket 4']
Calibration (Tuning): 2024-07-23 -> 2026-02-26


## 3. Gemeinsame Widget-Bausteine

`make_toggle_range_row` baut pro Parameter eine Zeile mit Checkbox ("tunen") + Fix-Wert-Feld
(aktiv wenn AUS) + Bereichs-/Auswahl-Feld (aktiv wenn AN). `collect_search_space` liest alle Zeilen
eines Panels aus und baut daraus `(fixed_overrides, search_space)` -- `fixed_overrides` wird per
`dataclasses.replace` auf `BASE_PARAMS` angewendet, `search_space` geht direkt an
`tune_basket_params` (siehe `Src.backtest.tuning.SearchSpec`: **tuple = Bereich, list = Auswahl** --
z.B. `wilks_p_content` unten nutzt bewusst eine Liste, kein Bereich, weil (90%, 95%) zwei diskrete
Optionen aus Lin et al. (2021) sind, kein Kontinuum).

`make_period_window_widgets`/`resolve_period_and_fold_days`: Zeitraum-Eingrenzung (Start-Datum +
Gesamtdauer in Tagen -- Default `DEFAULT_PERIOD_START` = 1. März 2025, `DEFAULT_PERIOD_DAYS` = 61
Tage, für zügige Tests; der Gesamtdauer-Slider lässt sich bis zur vollen `CALIBRATION_RANGE`-Länge
ab diesem Start hochziehen) und Fenster-**Anzahl** (`n_windows`) statt Fenster-**Größe** (`fold_days`
direkt) -- analog zum alten `tuning_common.auto_walkforward_layout`. Die neue Engine kennt nur
`fold_days`, `Src.backtest.tuning.fold_days_for_n_windows` rechnet das aus `n_windows` zurück.

In [4]:
def make_toggle_range_row(
    name: str, *, kind: str, fix_default, range_default, description: str,
    fix_min=None, fix_max=None, fix_step=None, range_min=None, range_max=None, range_step=None,
    options=None,
):
    """Baut (tune_checkbox, fix_widget, range_widget, kind, row_box) fuer EINEN
    BasketParams-Feldnamen. kind: 'int', 'float' oder 'categorical'.

    - tune_checkbox AUS (Default) -> fix_widget aktiv, Parameter landet in fixed_overrides.
    - tune_checkbox AN -> range_widget aktiv, Parameter landet in search_space
      (tuple fuer 'int'/'float' -- IMMER ein Bereich; list fuer 'categorical' --
      IMMER eine Auswahl, siehe Src.backtest.tuning.SearchSpec-Docstring).
    """
    tune_checkbox = widgets.Checkbox(value=False, description="tunen", indent=False, layout=widgets.Layout(width="90px"))

    if kind == "int":
        fix_widget = widgets.IntSlider(value=fix_default, min=fix_min, max=fix_max, step=fix_step or 1, description=name + ":", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
        range_widget = widgets.IntRangeSlider(value=range_default, min=range_min, max=range_max, step=range_step or 1, description=name + " (Bereich):", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"), disabled=True)
    elif kind == "float":
        fix_widget = widgets.FloatSlider(value=fix_default, min=fix_min, max=fix_max, step=fix_step or 0.01, description=name + ":", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
        range_widget = widgets.FloatRangeSlider(value=range_default, min=range_min, max=range_max, step=range_step or 0.01, description=name + " (Bereich):", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"), disabled=True)
    elif kind == "categorical":
        fix_widget = widgets.Dropdown(options=options, value=fix_default, description=name + ":", style={"description_width": "220px"})
        range_widget = widgets.SelectMultiple(options=options, value=tuple(range_default), description=name + " (Optionen):", style={"description_width": "220px"}, layout=widgets.Layout(width="460px", height="55px"), disabled=True)
    else:
        raise ValueError(f"Unbekannte kind={kind!r}")

    fix_widget.disabled = False

    def _on_toggle(change):
        range_widget.disabled = not change["new"]
        fix_widget.disabled = change["new"]

    tune_checkbox.observe(_on_toggle, names="value")

    desc_html = widgets.HTML(f"<span style='color:#888; font-size:0.85em;'>{description}</span>", layout=widgets.Layout(margin="8px 0 0 10px"))
    row = widgets.HBox([tune_checkbox, fix_widget, range_widget, desc_html])
    return tune_checkbox, fix_widget, range_widget, kind, row


def collect_search_space(param_widgets: dict) -> tuple[dict, dict]:
    """Liest alle make_toggle_range_row-Widgets aus und baut
    (fixed_overrides, search_space) fuer den tune_basket_params-Aufruf.
    tune_checkbox=AUS -> fixed_overrides (dataclasses.replace auf BASE_PARAMS).
    tune_checkbox=AN  -> search_space (tuple=Bereich bei int/float, list=Auswahl bei categorical).
    """
    fixed_overrides, search_space = {}, {}
    for name, (tune_checkbox, fix_widget, range_widget, kind, _row) in param_widgets.items():
        if tune_checkbox.value:
            search_space[name] = list(range_widget.value) if kind == "categorical" else tuple(range_widget.value)
        else:
            fixed_overrides[name] = fix_widget.value
    return fixed_overrides, search_space


#!#################### GEÄNDERT: Default-Zeitraum ist jetzt ein kurzer,
#!#################### fester Ausschnitt (1. Maerz 2025, 61 Tage) statt der
#!#################### vollen CALIBRATION_RANGE -- fuer zuegige Tests ohne
#!#################### jedes Mal manuell einschraenken zu muessen; der
#!#################### Gesamtdauer-Slider erlaubt weiterhin bis zur vollen
#!#################### CALIBRATION_RANGE ab DEFAULT_PERIOD_START.
DEFAULT_PERIOD_START = pd.Timestamp("2025-03-01", tz="UTC")
DEFAULT_PERIOD_DAYS = 61


def make_period_window_widgets():
    """Baut die drei Widgets fuer Zeitraum-Eingrenzung + Fenster-Anzahl:
    Start-Datum, Gesamtdauer (Tage), n_windows. Analog zum alten
    tuning_common.auto_walkforward_layout (dort: n_windows war die einzige
    Nutzer-Stellschraube fuer die Fenstergroesse) -- hier zusaetzlich mit
    einer Zeitraum-EINGRENZUNG (Start + Dauer statt immer der vollen
    CALIBRATION_RANGE), um schnelle Tests auf einem kurzen Ausschnitt zu
    erlauben, bevor ein Tuning auf dem vollen Zeitraum laeuft.

    Default: DEFAULT_PERIOD_START (1. Maerz 2025) + DEFAULT_PERIOD_DAYS
    (61 Tage) -- kurzer, zuegiger Testlauf; der Gesamtdauer-Slider laesst
    sich bis zur vollen CALIBRATION_RANGE-Laenge ab DEFAULT_PERIOD_START
    hochziehen, falls doch der ganze Zeitraum gebraucht wird.
    """
    max_days_from_default_start = (CALIBRATION_RANGE.end - DEFAULT_PERIOD_START).days + 1
    w_start = widgets.DatePicker(value=DEFAULT_PERIOD_START.date(), description="Start:", style={"description_width": "220px"})
    w_total_days = widgets.IntSlider(value=DEFAULT_PERIOD_DAYS, min=10, max=max_days_from_default_start, step=1, description="Gesamtdauer (Tage):", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
    w_n_windows = widgets.IntSlider(value=6, min=1, max=30, step=1, description="n_windows:", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
    return w_start, w_total_days, w_n_windows
#!####################


def resolve_period_and_fold_days(w_start, w_total_days, w_n_windows, *, base_params, strategy_name):
    """Liest (w_start, w_total_days, w_n_windows) aus und baut daraus
    (effective_calibration_range, fold_days) fuer tune_basket_params --
    effective_calibration_range ist auf [w_start, w_start+w_total_days-1]
    EINGESCHRAENKT (nie groesser als CALIBRATION_RANGE, siehe Clip unten),
    fold_days kommt aus Src.backtest.tuning.fold_days_for_n_windows.
    """
    start_ts = pd.Timestamp(w_start.value, tz="UTC")
    end_ts = min(start_ts + pd.Timedelta(days=w_total_days.value - 1), CALIBRATION_RANGE.end)
    start_ts = max(start_ts, CALIBRATION_RANGE.start)
    effective_range = bt_tuning.DateRange(start_ts, end_ts)

    fold_days = bt_tuning.fold_days_for_n_windows(
        effective_range, n_windows=w_n_windows.value, base_params=base_params, strategy_name=strategy_name,
    )
    return effective_range, fold_days


print("Widget-Bausteine bereit.")

Widget-Bausteine bereit.


## 4. Pairwise-Z-Score-Strategie (`pairwise_zscore`)

In [ ]:
from dataclasses import replace as _dc_replace

pw_w_basket = widgets.Dropdown(options=BASKET_LABELS, value=BASKET_LABELS[0], description="Basket:", style={"description_width": "220px"})
pw_w_start, pw_w_total_days, pw_w_n_windows = make_period_window_widgets()
pw_w_trials = widgets.IntSlider(value=20, min=5, max=200, step=5, description="n_trials:", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
pw_w_min_trades = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.5, description="min_trades_per_fold:", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
pw_w_fee = widgets.Dropdown(options=FEE_PRESETS, value="standard", description="Fee-Preset:", style={"description_width": "220px"})

pw_params = {
    "entry_z": make_toggle_range_row("entry_z", kind="float", fix_default=2.0, range_default=(1.0, 4.0), description="Entry-Schwelle (|Z-Score|).", fix_min=0.5, fix_max=5.0, fix_step=0.1, range_min=0.5, range_max=5.0, range_step=0.1),
    "exit_z": make_toggle_range_row("exit_z", kind="float", fix_default=0.5, range_default=(0.0, 1.5), description="Exit-Schwelle (|Z-Score|) -- bei Suche automatisch < entry_z gekappt.", fix_min=0.0, fix_max=3.0, fix_step=0.1, range_min=0.0, range_max=3.0, range_step=0.1),
    "max_holding_half_life_multiplier": make_toggle_range_row("max_holding_half_life_multiplier", kind="float", fix_default=4.0, range_default=(1.0, 8.0), description="Timeout als Vielfaches der Half-Life.", fix_min=1.0, fix_max=12.0, fix_step=0.5, range_min=1.0, range_max=12.0, range_step=0.5),
    "stop_z": make_toggle_range_row("stop_z", kind="float", fix_default=4.0, range_default=(3.0, 6.0), description="Stop-Loss-Schwelle (|Z-Score|).", fix_min=2.0, fix_max=8.0, fix_step=0.5, range_min=2.0, range_max=8.0, range_step=0.5),
}

pw_run_button = widgets.Button(description="Pairwise-Tuning starten", button_style="primary", icon="bolt")
pw_output = widgets.Output()
PAIRWISE_RESULT = {}


def _pw_on_run(_):
    with pw_output:
        clear_output(wait=True)
        fixed_overrides, search_space = collect_search_space(pw_params)
        base_params = _dc_replace(BasketParams(fee_bps=_resolve_fee_bps(pw_w_fee.value)), **fixed_overrides)
        basket = pw_w_basket.value

        effective_range, fold_days = resolve_period_and_fold_days(
            pw_w_start, pw_w_total_days, pw_w_n_windows, base_params=base_params, strategy_name="pairwise_zscore",
        )
        print(f"Basket: {basket} | fee_preset: {pw_w_fee.value} | fixed: {fixed_overrides} | search_space: {search_space}")
        print(f"Zeitraum: {effective_range.start.date()} -> {effective_range.end.date()} | "
              f"n_windows={pw_w_n_windows.value} -> fold_days={fold_days}\n")

        storage_path = bt_tuning.TUNING_DIR / "optuna_walkforward.db"
        bt_tuning.ensure_dashboard_running(storage_path, port=bt_tuning.DASHBOARD_PORTS["pairwise_zscore"])

        result = bt_tuning.tune_basket_params(
            basket=basket, panel=panel, timeline=timeline, base_params=base_params,
            calibration_range=effective_range, strategy_name="pairwise_zscore",
            fold_days=fold_days, n_trials=pw_w_trials.value,
            min_trades_per_fold=pw_w_min_trades.value, search_space=search_space or None,
            study_name=f"{RUN_LABEL_PREFIX}-pairwise_zscore-{basket}", storage_path=storage_path,
            progress=True,  #!#################### NEU -- ohne das bleibt tune_basket_params bis zum Ende stumm
        )
        print(f"\nFertig. Bestes Objective: {result.study.best_value:.4f} ueber {len(result.trials_df)} Trials, {len(result.folds)} Folds")
        print(f"Beste Parameter: {result.best_params}")
        display(result.trials_df.sort_values("objective", ascending=False).head(5))

        run_dir = bt_tuning.save_tuning_results({basket: result}, label=f"{RUN_LABEL_PREFIX}-pairwise_zscore-{basket}")
        print(f"Gespeichert: {run_dir}")
        PAIRWISE_RESULT[basket] = result


# _click_handlers.callbacks leeren, BEVOR on_click neu registriert wird -- sonst haengt sich bei
# jeder erneuten Ausfuehrung dieser Zelle (z.B. "Run All" nach einer Aenderung weiter oben) ein
# zusaetzlicher Handler an denselben Button, sodass EIN Klick mehrere parallele Laeufe ausloest.
pw_run_button._click_handlers.callbacks.clear()
pw_run_button.on_click(_pw_on_run)
display(widgets.VBox([
    widgets.HTML("<b>Pairwise-Z-Score (Src.backtest.strategy.PairwiseZScoreStrategy)</b>"),
    widgets.HTML("<u>Suche, Zeitraum & Basket</u>"),
    pw_w_basket, pw_w_start, pw_w_total_days, pw_w_n_windows, pw_w_trials, pw_w_min_trades, pw_w_fee,
    widgets.HTML("<u>Parameter</u> (Haekchen 'tunen' AN = Suche im Bereichsfeld, AUS = fix im linken Feld)"),
    *[p[4] for p in pw_params.values()],
    pw_run_button, pw_output,
]))

## 5. GARCH-Wilks-Toleranzintervall-Strategie (`garch_wilks`)

Deutlich teurer pro Trial (GARCH(1,1)-Fit pro Paar bei jedem neuen Clique-Snapshot) -- Default-Werte für `n_trials`/`fold_days` daher niedriger als bei Pairwise.

In [6]:
gw_w_basket = widgets.Dropdown(options=BASKET_LABELS, value=BASKET_LABELS[0], description="Basket:", style={"description_width": "220px"})
gw_w_start, gw_w_total_days, gw_w_n_windows = make_period_window_widgets()
gw_w_n_windows.value = 3  # GARCH-Fit ist teuer -- weniger Fenster als Default fuer Pairwise
gw_w_trials = widgets.IntSlider(value=8, min=3, max=60, step=1, description="n_trials:", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
gw_w_min_trades = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.5, description="min_trades_per_fold:", style={"description_width": "220px"}, layout=widgets.Layout(width="460px"))
gw_w_fee = widgets.Dropdown(options=FEE_PRESETS, value="standard", description="Fee-Preset:", style={"description_width": "220px"})

gw_params = {
    "wilks_lookback_candles": make_toggle_range_row("wilks_lookback_candles", kind="int", fix_default=288, range_default=(100, 400), description="Rollierendes Fenster fuer die Xmax-Historie der Wilks-Grenzen.", fix_min=50, fix_max=600, range_min=50, range_max=600),
    "wilks_p_content": make_toggle_range_row("wilks_p_content", kind="categorical", fix_default=0.95, range_default=[0.90, 0.95], description="Geforderter abgedeckter Anteil (Lin et al. 2021: 90% oder 95%).", options=[0.90, 0.95]),
    "wilks_confidence": make_toggle_range_row("wilks_confidence", kind="float", fix_default=0.95, range_default=(0.90, 0.99), description="Konfidenzniveau des Wilks-Toleranzintervalls.", fix_min=0.80, fix_max=0.99, fix_step=0.01, range_min=0.80, range_max=0.99, range_step=0.01),
    "max_holding_half_life_multiplier": make_toggle_range_row("max_holding_half_life_multiplier", kind="float", fix_default=4.0, range_default=(1.0, 8.0), description="Timeout als Vielfaches der Half-Life.", fix_min=1.0, fix_max=12.0, fix_step=0.5, range_min=1.0, range_max=12.0, range_step=0.5),
}

gw_run_button = widgets.Button(description="GARCH-Wilks-Tuning starten", button_style="primary", icon="bolt")
gw_output = widgets.Output()
GARCH_WILKS_RESULT = {}


def _gw_on_run(_):
    with gw_output:
        clear_output(wait=True)
        fixed_overrides, search_space = collect_search_space(gw_params)
        base_params = _dc_replace(BasketParams(fee_bps=_resolve_fee_bps(gw_w_fee.value)), **fixed_overrides)
        basket = gw_w_basket.value

        effective_range, fold_days = resolve_period_and_fold_days(
            gw_w_start, gw_w_total_days, gw_w_n_windows, base_params=base_params, strategy_name="garch_wilks",
        )
        print(f"Basket: {basket} | fee_preset: {gw_w_fee.value} | fixed: {fixed_overrides} | search_space: {search_space}")
        print(f"Zeitraum: {effective_range.start.date()} -> {effective_range.end.date()} | "
              f"n_windows={gw_w_n_windows.value} -> fold_days={fold_days}")
        print("Hinweis: GARCH-Fit pro Paar/Snapshot ist teuer -- ein einzelner Trial kann mehrere Minuten "
              "dauern, siehe Fortschrittszeilen pro Fold unten.\n")

        storage_path = bt_tuning.TUNING_DIR / "optuna_walkforward.db"
        bt_tuning.ensure_dashboard_running(storage_path, port=bt_tuning.DASHBOARD_PORTS["garch_wilks"])

        result = bt_tuning.tune_basket_params(
            basket=basket, panel=panel, timeline=timeline, base_params=base_params,
            calibration_range=effective_range, strategy_name="garch_wilks",
            fold_days=fold_days, n_trials=gw_w_trials.value,
            min_trades_per_fold=gw_w_min_trades.value, search_space=search_space or None,
            study_name=f"{RUN_LABEL_PREFIX}-garch_wilks-{basket}", storage_path=storage_path,
            progress=True,  #!#################### NEU -- ohne das bleibt tune_basket_params bis zum Ende stumm
        )
        print(f"\nFertig. Bestes Objective: {result.study.best_value:.4f} ueber {len(result.trials_df)} Trials, {len(result.folds)} Folds")
        print(f"Beste Parameter: {result.best_params}")
        display(result.trials_df.sort_values("objective", ascending=False).head(5))

        run_dir = bt_tuning.save_tuning_results({basket: result}, label=f"{RUN_LABEL_PREFIX}-garch_wilks-{basket}")
        print(f"Gespeichert: {run_dir}")
        GARCH_WILKS_RESULT[basket] = result


gw_run_button._click_handlers.callbacks.clear()
gw_run_button.on_click(_gw_on_run)
display(widgets.VBox([
    widgets.HTML("<b>GARCH-Wilks-Toleranzintervall (Src.backtest.strategy.GarchWilksToleranceStrategy)</b>"),
    widgets.HTML("<u>Suche, Zeitraum & Basket</u>"),
    gw_w_basket, gw_w_start, gw_w_total_days, gw_w_n_windows, gw_w_trials, gw_w_min_trades, gw_w_fee,
    widgets.HTML("<u>Parameter</u> (Haekchen 'tunen' AN = Suche im Bereichsfeld, AUS = fix im linken Feld)"),
    *[p[4] for p in gw_params.values()],
    gw_run_button, gw_output,
]))

## 6. Übersicht: in dieser Session getunte Parameter je Strategie/Basket

Zeigt `PAIRWISE_RESULT`/`GARCH_WILKS_RESULT` (in dieser Notebook-Session gelaufene Panels) -- für alle jemals gespeicherten Läufe siehe die `data/processed/tuning/<run_id>/best_params.json`-Ordner direkt, oder `06_strategy_comparison.ipynb` für den Out-of-Sample-Vergleich.

In [7]:
def show_session_overview():
    rows = []
    for strategy_key, results in (("pairwise_zscore", PAIRWISE_RESULT), ("garch_wilks", GARCH_WILKS_RESULT)):
        for basket, result in results.items():
            rows.append({
                "strategy": strategy_key, "basket": basket,
                "best_objective": result.study.best_value, "n_trials": len(result.trials_df),
                "best_params": result.best_params,
            })
    overview_df = pd.DataFrame(rows)
    if overview_df.empty:
        print("Noch keine getunten Parameter in dieser Session -- oben mindestens ein Panel ausfuehren.")
    else:
        display(overview_df)
    return overview_df


_ = show_session_overview()

Noch keine getunten Parameter in dieser Session -- oben mindestens ein Panel ausfuehren.
